# Example: Bandit Challenger and Validation Report

In this example, we implement the **epsilon-greedy combinatorial bandit** for adaptive asset selection, backtest it against the Cobb-Douglas engine and buy-and-hold across all HMM-generated paths, and produce a formal validation report with pass/fail criteria for deployment readiness.

> __Learning Objectives:__
>
> * __Implement and run the bandit portfolio selector:__ Build a bandit context and run the epsilon-greedy solver using [the `solve_bandit(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.solve_bandit) with [the `bandit_world(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.bandit_world) as the reward signal. Visualize convergence through reward history, exploration decay, and cumulative regret.
> * __Compare three strategies across HMM paths:__ Backtest the Cobb-Douglas engine, bandit selector, and buy-and-hold across 100 regime-switching paths using [the `backtest_bandit(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.backtest_bandit). Evaluate head-to-head performance using wealth, drawdown, Sharpe ratio, and failure rate metrics.
> * __Produce a validation report for production readiness:__ Apply explicit pass/fail criteria to each strategy using [the `build(MyValidationReport; ...)` factory method](../../code/docs/build/session3.html#eCornellAIFinance.build-Tuple{Type{MyValidationReport}}). Determine which strategies are cleared for Session 4 production deployment.

Let's dive in!

## Setup, Data and Prerequisites
We begin by loading the `eCornellAIFinance` package and the pre-computed backtest scenario and results from Example 1 via `Include.jl`.

In [ ]:
# Load the eCornellAIFinance package, dependencies, and path constants.
include("Include.jl");

In the next cell, we load the backtest scenario and engine results saved from Example 1 (HMM backtesting). The global variables `scenario::MyBacktestScenario`, `engine_bt::MyBacktestResult`, and `bh_bt::MyBacktestResult` store the scenario data and baseline results, respectively.

In [ ]:
let
    # --- Step 1: Load the backtest scenario and results from Example 1 ---
    # The scenario contains HMM-generated market and price paths.
    # The engine and buy-and-hold results were computed in Example 1.
    scenario_data = load(joinpath(_PATH_TO_DATA, "backtest-scenario.jld2"));
    global scenario = scenario_data["scenario"];

    results_data = load(joinpath(_PATH_TO_DATA, "backtest-results.jld2"));
    global engine_bt = results_data["engine"];
    global bh_bt = results_data["buyhold"];

    # --- Step 2: Define constants used throughout the notebook ---
    global Δt = 1.0 / 252.0;       # daily time step in years (1 trading day = 1/252 year)
    global rf = 0.05;               # annual risk-free rate
    global B₀ = 10000.0;            # initial portfolio budget ($)
    global N_short = 21;            # short EMA window (approx 1 month)
    global N_long = 63;             # long EMA window (approx 1 quarter)
    global offset = N_short + N_long;  # warm-up period before trading begins
    global n_mc_paths = scenario.n_paths;  # number of Monte Carlo paths

    # --- Step 3: Define the 5 synthetic asset tickers and their SIM parameters ---
    # Each ticker has parameters (alpha, beta, sigma) for the Single Index Model.
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    # --- Step 4: Print summary ---
    println("Loaded: $(n_mc_paths) paths, $(length(tickers)) tickers")
    println("Engine and buy-and-hold results loaded from Example 1")
end

### Implementation
The `strategy_metrics(bt::MyBacktestResult)` helper function computes six summary performance metrics from a backtest result: median final wealth, 10th and 90th percentile wealth, median max drawdown, median Sharpe ratio, and failure rate. It takes a single `MyBacktestResult` argument and returns a named tuple of rounded values.

In [ ]:
"""
    strategy_metrics(bt::MyBacktestResult) -> NamedTuple

Compute six summary performance metrics from a backtest result across all Monte Carlo paths.

### Arguments
- `bt::MyBacktestResult`: A backtest result containing `final_wealth`, `max_drawdowns`, and `sharpe_ratios` fields.

### Returns
A named tuple with fields:
- `med_wealth`: Median final portfolio value (rounded to integer).
- `p10_wealth`: 10th percentile final wealth (downside).
- `p90_wealth`: 90th percentile final wealth (upside).
- `med_dd`: Median max drawdown in percent.
- `med_sharpe`: Median Sharpe ratio.
- `fail_rate`: Percentage of paths with final wealth below `B₀`.
"""
function strategy_metrics(bt::MyBacktestResult)
    return (
        med_wealth = round(median(bt.final_wealth), digits=0),          # median final portfolio value
        p10_wealth = round(quantile(bt.final_wealth, 0.10), digits=0),  # 10th percentile (downside)
        p90_wealth = round(quantile(bt.final_wealth, 0.90), digits=0),  # 90th percentile (upside)
        med_dd = round(median(bt.max_drawdowns) * 100, digits=1),       # median max drawdown (%)
        med_sharpe = round(median(bt.sharpe_ratios), digits=3),         # median Sharpe ratio
        fail_rate = round(sum(bt.final_wealth .< B₀) / length(bt.final_wealth) * 100, digits=1) # % paths losing money
    )
end;

___
## Task 1: Implement and Run the Bandit Portfolio Selector
We demonstrate the epsilon-greedy bandit on a single representative path. The bandit explores $2^K - 1 = 31$ possible asset subsets, using the Cobb-Douglas utility as its reward signal via [the `bandit_world(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.bandit_world). We visualize how the bandit converges to a preferred subset.

> __What should you see?__
>
> The reward history should show high variance initially (exploration) and stabilize as the bandit converges (exploitation). The exploration probability $\epsilon_t$ decays over time. The winning arm should be a sensible asset subset given the market conditions.

In the next cell, we build a `MyBanditContext` for the first path, run [the `solve_bandit(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.solve_bandit), and store the output in the `bandit_result::MyBanditResult` variable.

In [ ]:
let
    # --- Step 1: Extract data for the first path as a representative example ---
    K = length(tickers);                       # number of assets (K = 5)
    mkt = scenario.market_paths[1, :];         # market index path 1

    # --- Step 2: Compute the sentiment signal λ at the offset point ---
    # λ is derived from the crossover of short and long EMAs of the market index.
    # High λ = bullish sentiment (short EMA above long EMA); low λ = bearish.
    ema_s = compute_ema(mkt; window=N_short);  # short-window EMA (21 days)
    ema_l = compute_ema(mkt; window=N_long);   # long-window EMA (63 days)
    λ = compute_lambda(ema_s, ema_l; G=10.0);  # sentiment signal with gain G=10

    # --- Step 3: Compute the smoothed market growth rate at the offset ---
    gm_raw = compute_market_growth(mkt; Δt=Δt);       # raw daily market growth
    gm_e = compute_ema(gm_raw; window=10);             # smoothed with 10-day EMA

    # --- Step 4: Extract current prices for each asset at the offset time step ---
    prices_at_offset = [scenario.price_paths[1, offset, k] for k in 1:K];

    # --- Step 5: Build the bandit context (static inputs for the world function) ---
    # The context tells the bandit what assets, prices, budget, and market state to use.
    bandit_ctx = build(MyBanditContext, (
        tickers = tickers, sim_parameters = sim_params,
        prices = prices_at_offset, B = B₀,
        gm_t = gm_e[min(offset, length(gm_e))],
        lambda = λ[offset], epsilon = 0.1
    ));

    # --- Step 6: Configure and run the epsilon-greedy bandit solver ---
    # K=5 assets means 2^5 - 1 = 31 possible arms (all non-empty subsets).
    # The solver explores arms using decaying epsilon and averages rewards with learning rate alpha.
    bandit_model = build(MyEpsilonGreedyBanditModel, (
        K = K, n_iterations = 500, alpha = 0.1
    ));

    global bandit_result = solve_bandit(bandit_model, bandit_ctx);

    # --- Step 7: Report the best arm found by the bandit ---
    println("Bandit converged after 500 iterations:")
    println("  Best action: $(bandit_result.best_action)")
    selected_tickers = tickers[bandit_result.best_action .== 1];
    excluded_tickers = tickers[bandit_result.best_action .== 0];
    println("  Selected assets: $(selected_tickers)")
    println("  Excluded assets: $(excluded_tickers)")
    println("  Best utility: $(round(bandit_result.best_utility, digits=4))")
end

**Visualize:** Bandit convergence (reward history and exploration decay).

In the next cell, we plot the reward history with a running average and the exploration probability $\epsilon_t$ over iterations.

In [ ]:
let
    # --- Step 1: Extract iteration indices from the bandit result ---
    T_bandit = length(bandit_result.reward_history);
    iters = 1:T_bandit;

    # --- Step 2: Compute a running average of the reward signal ---
    # A 20-iteration moving average smooths the noisy per-iteration rewards
    # to reveal the convergence trend.
    window = 20;
    running_avg = [mean(bandit_result.reward_history[max(1, t-window+1):t]) for t in 1:T_bandit];

    # --- Step 3: Plot reward history (top panel) ---
    # Scatter shows individual rewards; line shows the smoothed trend.
    # Early iterations should be noisy (exploration); later ones stable (exploitation).
    p1 = scatter(iters, bandit_result.reward_history, label="Reward", alpha=0.2,
        markersize=2, color=:steelblue,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)
    plot!(p1, iters, running_avg, label="Running avg ($(window))", linewidth=2, color=:coral)
    ylabel!(p1, "Cobb-Douglas Utility")
    title!(p1, "Bandit Reward History")

    # --- Step 4: Plot exploration probability decay (bottom panel) ---
    # εₜ decays as t^(-1/3) * (K * log(t))^(1/3), balancing exploration vs exploitation.
    p2 = plot(iters, bandit_result.exploration_history, label="εₜ", linewidth=2, color=:purple,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)
    xlabel!(p2, "Iteration")
    ylabel!(p2, "Exploration Prob.")
    title!(p2, "Exploration Decay")

    # --- Step 5: Combine into a two-panel layout ---
    plot(p1, p2, layout=(2, 1), size=(800, 500), legend=:topright)
end

**Visualize:** Cumulative regret, which measures how much reward was lost to exploration.

In the next cell, we compute cumulative regret using [the `compute_regret(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.compute_regret) and plot it over iterations.

In [ ]:
let
    # --- Step 1: Compute cumulative regret from the reward history ---
    # Regret measures the gap between the best-in-hindsight reward and the actual reward.
    # A sublinear regret curve (concave shape) indicates the bandit is learning.
    regret = compute_regret(bandit_result.reward_history);

    # --- Step 2: Plot cumulative regret over iterations ---
    plot(1:length(regret), regret, label="Cumulative Regret", linewidth=2, color=:coral,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)
    xlabel!("Iteration")
    ylabel!("Cumulative Regret")
    title!("Bandit Cumulative Regret")
    plot!(size=(800, 350))
end

___
## Task 2: Backtest Bandit-Selected Portfolios Across All Paths
We run the bandit selector across all HMM-generated paths and compare three strategies head-to-head: the Cobb-Douglas engine (all assets, utility allocation), the bandit selector (learned asset subset + Cobb-Douglas allocation), and buy-and-hold.

> __What should you see?__
>
> A comparison table with 6 metrics across 3 strategies. The bandit may outperform if it successfully excludes assets that hurt during regime shifts. It may underperform if its initial learning phase costs too much. The key question: does adaptive asset _selection_ add value over adaptive asset _weighting_?

In the next cell, we run [the `backtest_bandit(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.backtest_bandit) across all paths and store the output in the `bandit_bt::MyBacktestResult` variable.

In [ ]:
let
    # --- Step 1: Run the bandit backtest across all Monte Carlo paths ---
    # For each path, the bandit selects the best asset subset, then applies
    # Cobb-Douglas allocation over the selected assets for the remaining trading days.
    println("Backtesting Bandit Selector across $(n_mc_paths) paths...")
    global bandit_bt = backtest_bandit(scenario, tickers, sim_params;
        B₀=B₀, offset=offset, n_bandit_iters=200);

    # --- Step 2: Print summary statistics ---
    println("Bandit backtest complete.")
    println("Median final wealth: \$$(round(median(bandit_bt.final_wealth), digits=0))")
end

**Head-to-head comparison table.**

In the next cell, we compute six performance metrics for each strategy and display a comparison table using `pretty_table`.

In [ ]:
let
    # --- Step 1: Compute metrics for all three strategies ---
    eng = strategy_metrics(engine_bt);
    ban = strategy_metrics(bandit_bt);
    bh = strategy_metrics(bh_bt);

    # --- Step 2: Build a comparison DataFrame ---
    comparison = DataFrame(
        "Metric" => ["Median Final Wealth", "10th Percentile Wealth",
            "90th Percentile Wealth", "Median Max Drawdown (%)",
            "Median Sharpe Ratio", "Failure Rate (%)"],
        "Cobb-Douglas Engine" => [eng.med_wealth, eng.p10_wealth, eng.p90_wealth,
            eng.med_dd, eng.med_sharpe, eng.fail_rate],
        "Bandit Selector" => [ban.med_wealth, ban.p10_wealth, ban.p90_wealth,
            ban.med_dd, ban.med_sharpe, ban.fail_rate],
        "Buy-and-Hold" => [bh.med_wealth, bh.p10_wealth, bh.p90_wealth,
            bh.med_dd, bh.med_sharpe, bh.fail_rate]
    );

    # --- Step 3: Display the formatted table ---
    println("═"^70)
    println("  HEAD-TO-HEAD: 3 Strategies × $(n_mc_paths) HMM Paths")
    println("═"^70)
    pretty_table(comparison, tf=tf_markdown)
end

**Visualize:** Box plots showing final wealth and max drawdown distributions across strategies.

In the next cell, we plot side-by-side box plots comparing the three strategies on final wealth and max drawdown.

In [ ]:
let
    # --- Step 1: Box plot of final wealth distributions (left panel) ---
    # The dashed line at B₀ marks the break-even threshold.
    # Paths below B₀ represent losses (failure rate).
    p1 = boxplot(["CD Engine" "Bandit" "Buy-Hold"],
        [engine_bt.final_wealth bandit_bt.final_wealth bh_bt.final_wealth],
        legend=false, ylabel="Final Wealth (\$)", title="Final Wealth Distribution",
        color=[:steelblue :green :coral], alpha=0.7,
        bg="gray95", background_color_outside="white", framestyle=:box)
    hline!(p1, [B₀], linestyle=:dash, color=:black, alpha=0.5, label="")

    # --- Step 2: Box plot of max drawdown distributions (right panel) ---
    # Lower drawdown is better. Shows how much each strategy lost at its worst point.
    p2 = boxplot(["CD Engine" "Bandit" "Buy-Hold"],
        [engine_bt.max_drawdowns.*100 bandit_bt.max_drawdowns.*100 bh_bt.max_drawdowns.*100],
        legend=false, ylabel="Max Drawdown (%)", title="Max Drawdown Distribution",
        color=[:steelblue :green :coral], alpha=0.7,
        bg="gray95", background_color_outside="white", framestyle=:box)

    # --- Step 3: Combine into a side-by-side layout ---
    plot(p1, p2, layout=(1, 2), size=(900, 400))
end

___
## Task 3: Formal Validation Report
We evaluate each strategy against explicit pass/fail criteria for production deployment:

| Criterion | Threshold |
|-----------|----------|
| Median Sharpe | $\geq$ 0.3 |
| Median Max Drawdown | $\leq$ 25% |
| Failure Rate | $\leq$ 10% |
| Beats Buy-and-Hold | Median wealth $>$ buy-and-hold median |

Strategies that pass all four criteria are cleared for Session 4 production deployment.

> __What should you see?__
>
> A pass/fail table for each strategy. The Cobb-Douglas engine should pass most criteria (it has built-in risk controls). The bandit's performance depends on whether adaptive selection adds value in this scenario. Buy-and-hold will likely fail on drawdown or failure rate.

In the next cell, we build a `MyValidationReport` for each strategy using [the `build(MyValidationReport; ...)` factory method](../../code/docs/build/session3.html#eCornellAIFinance.build-Tuple{Type{MyValidationReport}}) and print the detailed pass/fail results.

In [ ]:
let
    # --- Step 1: Compute buy-and-hold median wealth as the benchmark ---
    bh_median_wealth = median(bh_bt.final_wealth);

    # --- Step 2: Define the four pass/fail criteria thresholds ---
    # A strategy must pass all four to be cleared for production.
    criteria = Dict(
        "min_sharpe" => 0.3,             # Sharpe ratio must be at least 0.3
        "max_drawdown" => 0.25,          # max drawdown must not exceed 25%
        "max_failure_rate" => 0.10,      # at most 10% of paths can lose money
        "min_wealth_vs_bh" => bh_median_wealth  # must beat buy-and-hold median
    );

    # --- Step 3: Build validation reports for each strategy ---
    # The build() factory compares actuals to criteria and populates the passed dict.
    strategies = [
        ("Cobb-Douglas Engine", engine_bt),
        ("Bandit Selector", bandit_bt),
        ("Buy-and-Hold", bh_bt)
    ];

    global reports = [];
    for (label, bt) in strategies
        # Compute actual values for each criterion from the backtest results
        actuals = Dict(
            "min_sharpe" => median(bt.sharpe_ratios),
            "max_drawdown" => median(bt.max_drawdowns),
            "max_failure_rate" => sum(bt.final_wealth .< B₀) / length(bt.final_wealth),
            "min_wealth_vs_bh" => median(bt.final_wealth)
        );

        # Build the validation report (compares actuals vs criteria)
        report = build(MyValidationReport;
            strategy_label=label, criteria=criteria, actuals=actuals);
        push!(reports, report);
    end

    # --- Step 4: Display detailed validation results for each strategy ---
    println("═"^70)
    println("  VALIDATION REPORT: Production Readiness")
    println("═"^70)

    criterion_names = Dict(
        "min_sharpe" => "Sharpe ≥ 0.3",
        "max_drawdown" => "Max DD ≤ 25%",
        "max_failure_rate" => "Failure ≤ 10%",
        "min_wealth_vs_bh" => "Beats Buy-Hold"
    );

    for report in reports
        println("\n  Strategy: $(report.strategy_label)")
        println("  " * "-"^40)

        all_passed = true;
        for (key, passed) in report.passed
            status = passed ? "PASS" : "FAIL";
            actual = round(report.actuals[key], digits=3);
            threshold = round(report.criteria[key], digits=3);
            name = get(criterion_names, key, key);
            println("    $(name): $(status) (actual=$(actual), threshold=$(threshold))")
            if !passed
                all_passed = false;
            end
        end
        verdict = all_passed ? "CLEARED for Session 4" : "NOT CLEARED";
        println("  Verdict: $(verdict)")
    end
end

**Summary table:** Pass/fail at a glance.

In the next cell, we build a consolidated pass/fail table with an overall verdict row for each strategy.

In [ ]:
let
    # --- Step 1: Define the criterion keys and display labels ---
    criterion_keys = ["min_sharpe", "max_drawdown", "max_failure_rate", "min_wealth_vs_bh"];
    criterion_labels = ["Sharpe ≥ 0.3", "Max DD ≤ 25%", "Failure ≤ 10%", "Beats Buy-Hold"];

    # --- Step 2: Build rows from each report's pass/fail results ---
    rows = [];
    for (i, label) in enumerate(criterion_labels)
        row = [label];
        for report in reports
            key = criterion_keys[i];
            passed = get(report.passed, key, false);
            push!(row, passed ? "PASS" : "FAIL");
        end
        push!(rows, row);
    end

    # --- Step 3: Add an overall verdict row (CLEARED only if all criteria pass) ---
    verdict_row = ["OVERALL"];
    for report in reports
        all_pass = all(values(report.passed));
        push!(verdict_row, all_pass ? "CLEARED" : "NOT CLEARED");
    end
    push!(rows, verdict_row);

    # --- Step 4: Display the consolidated table ---
    data = reduce(hcat, rows)';
    pretty_table(data,
        header=["Criterion", [r.strategy_label for r in reports]...],
        alignment=[:l, :c, :c, :c])
end

___
## Summary
The epsilon-greedy bandit provides an adaptive challenger to the Cobb-Douglas engine by learning which asset subsets maximize utility through exploration and exploitation.

> __Key Takeaways:__
>
> * **Bandit exploration converges to preferred asset subsets:** The epsilon-greedy bandit tests all possible asset combinations and converges to the subset with the highest Cobb-Douglas utility. Reward variance decreases as exploration decays, and cumulative regret grows sublinearly.
> * **Head-to-head backtesting reveals the value of adaptive selection:** Comparing three strategies across 100 HMM paths shows whether adaptive asset selection (bandit) adds value over adaptive asset weighting (Cobb-Douglas engine). The comparison uses wealth, drawdown, Sharpe ratio, and failure rate metrics.
> * **Validation reports enforce deployment readiness:** Explicit pass/fail criteria (Sharpe, drawdown, failure rate, and benchmark comparison) determine which strategies are production-ready. Only strategies that pass all four criteria are cleared for Session 4 deployment.

In Session 4, the bandit and Cobb-Douglas allocator will work together: the bandit selects assets continuously, and the allocator decides how much of each. We will add sentiment monitoring, alert systems, and human override protocols.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.